# 🚀 Apache Airflow XCom — Complete Guide (Manual vs Automatic + Code)

---

# 🧠 1. What is XCom?

XCom = Cross Communication

Used to:
- Pass data between tasks in a DAG

---

# 🎯 Key Idea

Tasks in Airflow are isolated
XCom allows them to share small data

---

# ⚠️ Important Rules

- Only for small data
- Stored in metadata DB (xcom table)
- Works per task instance

---

# 🎯 2. Two Ways to Use XCom

---

# 🔥 2.1 Manual XCom (Classic Way)

## Using with DAG

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def push_data(ti):
    ti.xcom_push(key="my_key", value="hello")

def pull_data(ti):
    value = ti.xcom_pull(task_ids="push_task", key="my_key")
    print(f"Received: {value}")

with DAG(
    dag_id="xcom_manual_classic",
    start_date=datetime(2024,1,1),
    schedule="@daily",
    catchup=False
) as dag:

    push_task = PythonOperator(
        task_id="push_task",
        python_callable=push_data
    )

    pull_task = PythonOperator(
        task_id="pull_task",
        python_callable=pull_data
    )

    push_task >> pull_task
```

---

# 🔥 2.2 Automatic XCom (Decorator Way)

```python
from airflow.decorators import dag, task
from datetime import datetime

@dag(start_date=datetime(2024,1,1), schedule="@daily", catchup=False)
def xcom_auto():

    @task
    def extract():
        return "hello"

    @task
    def transform(data):
        return data.upper()

    @task
    def load(data):
        print(f"Final: {data}")

    data = extract()
    transformed = transform(data)
    load(transformed)

dag = xcom_auto()
```

---

# ⚡ 3. Comparison

- Manual → explicit push/pull
- Automatic → return-based

---

# 🎯 4. Summary

XCom allows tasks to share small data between each other.

---

# 🔥 End


# 🚀 Apache Airflow XCom — Advanced Guide (Patterns + DB + Debugging)

---

# 🧠 1. Recap

XCom = Cross Communication
Used to pass small data between tasks

---

# 🔥 2. Advanced Patterns

---

## 🔹 Multiple Outputs (Decorator)

```python
from airflow.decorators import task

@task
def extract():
    return {"name": "john", "age": 30}

@task
def process(data):
    print(data["name"], data["age"])
```

---

## 🔹 Using XCom in Templates

```python
bash_command="echo {{ ti.xcom_pull(task_ids='task1') }}"
```

---

## 🔹 Push Multiple Keys (Manual)

```python
ti.xcom_push(key="k1", value=10)
ti.xcom_push(key="k2", value=20)
```

---

# 📊 3. XCom in Metadata DB

Table: xcom

Columns:
- dag_id
- task_id
- key
- value
- execution_date

---

## 🔹 Query Example

```sql
SELECT * FROM xcom;
```

---

# ⚙️ 4. How XCom Works Internally

1. Task returns value
2. Airflow serializes it
3. Stores in xcom table
4. Next task pulls it

---

# ⚠️ 5. Debugging XCom Issues

---

## ❌ Value is None

- Wrong task_id
- Wrong key

---

## ❌ Serialization Error

- Non-serializable object

Fix:
- Convert to JSON / string

---

## ❌ Large Data Issue

- DB overload

Fix:
- Use S3 / external storage

---

# 🔥 6. Best Practices

- Keep data small
- Use decorator style
- Avoid heavy objects
- Use IDs instead of data

---

# 🎯 7. Real Production Pattern

```python
@task
def get_file():
    return "file.csv"

@task
def process(file):
    print(file)
```

---

# 💥 Final One-Line

XCom is powerful for lightweight communication but should be used carefully to avoid performance issues.

---

# 🔥 END


# 🚀 Apache Airflow XCom — Ultimate Guide (End-to-End + Custom Backend + Real Debugging)

---

# 🧠 1. End-to-End DAG Example (API → Process → DB)

```python
from airflow.decorators import dag, task
from datetime import datetime
import requests

@dag(start_date=datetime(2024,1,1), schedule="@daily", catchup=False)
def xcom_pipeline():

    @task
    def fetch_api():
        data = {"id": 1, "name": "john"}
        return data

    @task
    def process(data):
        data["name"] = data["name"].upper()
        return data

    @task
    def load(data):
        print(f"Loading to DB: {data}")

    load(process(fetch_api()))

dag = xcom_pipeline()
```

---

# 🔥 2. Custom XCom Backend (S3 Concept)

---

## Why?

Default XCom stores data in DB → not good for large data

---

## Approach

- Store data in S3
- Save only reference in XCom

---

## Example Pattern

```python
@task
def upload_to_s3():
    file_path = "/tmp/data.csv"
    # upload to S3 (pseudo)
    return "s3://bucket/data.csv"
```

---

## Next Task

```python
@task
def process(file_path):
    print(f"Reading from {file_path}")
```

---

---

# ⚙️ 3. Real Debugging Scenarios

---

## 🔴 Scenario 1: XCom is None

Cause:
- Wrong task_id

Fix:
- Check upstream task name

---

## 🔴 Scenario 2: JSON Serialization Error

Cause:
- Object not serializable

Fix:
```python
return str(data)
```

---

## 🔴 Scenario 3: Large Data Crash

Cause:
- Huge data in XCom

Fix:
- Use S3 / DB instead

---

## 🔴 Scenario 4: Wrong Execution Order

Cause:
- Dependency missing

Fix:
```python
task1 >> task2
```

---

---

# 📊 4. Debug Using Metadata DB

---

## Query XCom

```sql
SELECT dag_id, task_id, key, value
FROM xcom
ORDER BY execution_date DESC;
```

---

---

# 🧠 5. Production Best Practices

---

- Never store large data
- Use external storage (S3, DB)
- Use decorator-based XCom
- Keep payload small

---

---

# 🚀 6. Advanced Pattern (Multiple Outputs)

```python
@task
def extract():
    return {"file": "data.csv", "count": 100}

@task
def process(data):
    print(data["file"], data["count"])
```

---

---

# 🎯 7. Key Takeaways

- XCom = lightweight communication
- Stored in metadata DB
- Automatic in decorator DAGs
- Avoid misuse

---

# 💥 One-Line Summary

XCom should be used for passing small metadata between tasks, while large data should be handled using external storage systems.

---

# 🔥 END
